# 03 - Build the database and export the dataset

Downloads the shards from the private Hub repo `Tsomaros/llm-movies-raw`, builds the SQLite database on the local disk, exports the parquet dataset and writes the data report, which goes back to the repo. No copy of the database is kept: it rebuilds from the shards.
Use a **CPU runtime**. Needs the Colab Secret `HF_TOKEN` (write access).

**To resume: rerun this notebook.** Everything is rebuilt from the repo each time (deterministic).
The last cell is **one chain** (`&&`): a failed download stops it before the build, so a thin database can never be exported or pushed.
**After a change to `config/data.yaml`** (years, `base_model_cutoff`, `min_votes`...) run it again: the setup cells pull the new config, and `export` stops on a split that no longer matches, so the database needs a rebuild first.

**`PUSH`** uploads the dataset to the Hub repo `Tsomaros/llm-movies-data`. That repo must be **private or not exist yet**: the export refuses a public repo, because `locked.parquet` holds the test labels. Leave `PUSH` off for a dry run.

## Setup

Run these cells top to bottom. They are the same in all three notebooks.

In [ ]:
# clone the repo the first time, update it after that
!test -d /content/LLM_Movies_Model/.git && git -C /content/LLM_Movies_Model pull || git clone https://github.com/Tsomaros/LLM_Movies_Model /content/LLM_Movies_Model
%cd /content/LLM_Movies_Model

In [ ]:
!pip install -q uv
!uv sync --frozen

## Secrets and environment

Reads `TMDB_READ_TOKEN` and `HF_TOKEN` (write access) from Colab Secrets, with notebook access switched on. They are never printed. `UV_FROZEN` stops later `uv run` calls from re-resolving the environment.

In [ ]:
import os

from google.colab import userdata

os.environ["TMDB_READ_TOKEN"] = userdata.get("TMDB_READ_TOKEN")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["UV_FROZEN"] = "1"

## Push switch

Off by default.

In [ ]:
PUSH = False  #@param {type:"boolean"}
PUSH_FLAG = "--push" if PUSH else ""

## Download, build, export, report

The report is uploaded to the repo as `reports/data_summary.md`. Commit it to the project repo by hand as `reports/data_summary.md`.

In [ ]:
!uv run python -m llm_movies.data.hub_store download --dir /content/raw && uv run python -m llm_movies.data.build_db --tmdb /content/raw/tmdb --wd-films /content/raw/wd_films --wd-people /content/raw/wd_people --out /content/build/movies.db && uv run python -m llm_movies.data.export --db /content/build/movies.db --out /content/build/dataset {PUSH_FLAG} && uv run python -m llm_movies.data.report --db /content/build/movies.db --parquet /content/build/dataset/films.parquet --out /content/build/reports/data_summary.md && uv run python -m llm_movies.data.hub_store upload --dir /content/build/reports --path-in-repo reports